# Notebook 16 — Self-Quiz: Chapter 6 (Encodings)

**20 questions covering chapter 6.** Auto-scored, base64-hidden answers.

## Sections

- **A** — Cardinality (Q1–2, Q13–14)
- **B** — Integer encoding β (Q3–4)
- **C** — Pair encoding φ (Q5–7)
- **D** — Lists and tuples (Q8)
- **E** — Program encoding (Q9–12)
- **F** — Properties + states (Q15–20)

(Sections deliberately interleaved for variety.)


In [1]:
import sys, json, base64
from pathlib import Path

for candidate in [Path.cwd(), Path.cwd() / 'notebooks', Path.cwd().parent / 'notebooks']:
    if (candidate / 'while_lang.py').exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise ImportError("could not find while_lang.py")

_RESULTS = {}

_KEY_B64 = (
    "eyIxIjogeyJhIjogIkIiLCAiZSI6ICJUaGVvcmVtIDI1IChDYW50b3IgZGlhZ29uYWwpOiBzdXBwb3NlIEZ1bijihJUsIOKElSkg"
    "d2VyZSBjb3VudGFibGU7IGxpc3QgaXRzIGVsZW1lbnRzIGbigoAsIGbigoEsIC4uLi4gRGVmaW5lIGcobikgPSBmX24obikgKyAx"
    "LiBUaGVuIGcgZGlmZmVycyBmcm9tIGVhY2ggZl9uIGF0IHBvc2l0aW9uIG4sIHNvIGcgaXNuJ3Qgb24gdGhlIGxpc3Qg4oCUIGNv"
    "bnRyYWRpY3Rpb24uIEhlbmNlIHVuY291bnRhYmxlLiJ9LCAiMiI6IHsiYSI6ICJDIiwgImUiOiAiV2hpbGUgcHJvZ3JhbXMgYXJl"
    "IGZpbml0ZSBzdHJpbmdzIG92ZXIgYSBmaW5pdGUgYWxwaGFiZXQg4oCUIGNvdW50YWJseSBtYW55IHN1Y2ggc3RyaW5ncyBleGlz"
    "dCAoY291bnRhYmlsaXR5IG9mIGZpbml0ZS1zdHJpbmcgbGFuZ3VhZ2VzKS4gQ29tYmluZWQgd2l0aCBUaGVvcmVtIDI1LCBtb3N0"
    "IGZ1bmN0aW9ucyDihJUg4oaSIOKElSBhcmUgdW5jb21wdXRhYmxlLiJ9LCAiMyI6IHsiYSI6ICJCIiwgImUiOiAizrIoeCkgPSAy"
    "eCBpZiB4IOKJpSAwLCBlbHNlIC0yeCAtIDEuIE5vbi1uZWdhdGl2ZXMgZ28gdG8gZXZlbiBuYXR1cmFscywgbmVnYXRpdmVzIHRv"
    "IG9kZC4gWmlnemFnOiAw4oamMCwgLTHihqYxLCAx4oamMiwgLTLihqYzLCAy4oamNCwgLi4uIn0sICI0IjogeyJhIjogNSwgImUi"
    "OiAizrIoLTMpID0gLTLCtygtMykgLSAxID0gNiAtIDEgPSA1LiJ9LCAiNSI6IHsiYSI6ICJDIiwgImUiOiAiz4YobSwgbikgPSAy"
    "Xm0gKDJuICsgMSkgLSAxLiBUaGUgZXhwb25lbnQgMl5tIGZvbGxvd2VkIGJ5IGFuIG9kZCBudW1iZXIgMm4rMSwgbWludXMgMSB0"
    "byBmaXQgaW4g4oSVIHN0YXJ0aW5nIGZyb20gMC4ifSwgIjYiOiB7ImEiOiAxMjc1LCAiZSI6ICI0MDgxNSArIDEgPSA0MDgxNiA9"
    "IDJeNCDCtyAyNTUxLiBTbyBtID0gNCwgMm4rMSA9IDI1NTEsIG4gPSAxMjc1LiBEZWNvZGVkIHBhaXI6ICg0LCAxMjc1KS4ifSwg"
    "IjciOiB7ImEiOiAiQyIsICJlIjogIlR1cGxlcyBlbmNvZGUgdmlhIHJpZ2h0LWZvbGQgb2Ygz4Y6ICh4LCB5LCB6KSDihqYgz4Yo"
    "eCwgz4YoeSwgeikpLiBUbyBkZWNvZGUgYSB0cmlwbGUsIGFwcGx5IM+GJyB0d2ljZSDigJQgZmlyc3QgcGVlbGluZyBvZmYgeCwg"
    "dGhlbiB5LCBsZWF2aW5nIHouIn0sICI4IjogeyJhIjogIkIiLCAiZSI6ICLPiChbXSkgPSAwOyDPiChuIDo6IGwpID0gz4Yobiwg"
    "z4gobCkpICsgMS4gVGhlICsxIGVuc3VyZXMgdGhlIGVtcHR5IGxpc3QgKDApIGlzIGRpc3Rpbmd1aXNoYWJsZSBmcm9tIM+GKDAs"
    "IDApID0gMC4ifSwgIjkiOiB7ImEiOiAiQyIsICJlIjogIs+GX1Moc2tpcCkgPSAwIGJ5IGRlZmluaXRpb24uIFRoZSBzbWFsbGVz"
    "dCBwb3NzaWJsZSBHw7ZkZWwgbnVtYmVyIGZvciBhIFdoaWxlIHByb2dyYW0gaXMgMCDigJQgcmVzZXJ2ZWQgZm9yIHNraXAuIn0s"
    "ICIxMCI6IHsiYSI6ICJBIiwgImUiOiAiz4ZfUyh4X2kgOj0gYSkgPSAxICsgNCDCtyDPhijPhl9WKHhfaSksIM+GX0EoYSkpLiBS"
    "ZXNpZHVlIDEgbW9kIDQgaW5kaWNhdGVzIGFuIGFzc2lnbm1lbnQ7IHRoZSBxdW90aWVudCBlbmNvZGVzICh2YXJpYWJsZSwgZXhw"
    "cmVzc2lvbikuIn0sICIxMSI6IHsiYSI6ICJCIiwgImUiOiAiz4ZfQShudW1lcmFsIG4pID0gNSDCtyDOsihuKS4gSGVyZSDOsiBo"
    "YW5kbGVzIHRoZSBpbnRlZ2VyLXRvLW5hdHVyYWwgbWFwcGluZyAoc2luY2UgbnVtZXJhbHMgY2FuIGJlIG5lZ2F0aXZlKTsgcmVz"
    "aWR1ZSAwIG1vZCA1IGluZGljYXRlcyBhIG51bWVyYWwuIn0sICIxMiI6IHsiYSI6ICJDIiwgImUiOiAiVGhlIGVuY29kaW5nIGlz"
    "IG1hdGhlbWF0aWNhbGx5IGNvcnJlY3QgYnV0IHByb2R1Y2VzIGFzdHJvbm9taWNhbGx5IGxhcmdlIG51bWJlcnMg4oCUIGV2ZW4g"
    "c2ltcGxlIHdoaWxlIGxvb3BzIGdpdmUgMTAwMCsgZGlnaXQgR8O2ZGVsIG51bWJlcnMuIEl0J3MgYSB0aGVvcmV0aWNhbCBleGlz"
    "dGVuY2UgcHJvb2YsIG5vdCBhIHNlcmlhbGl6YXRpb24gZm9ybWF0LiJ9LCAiMTMiOiB7ImEiOiAiQiIsICJlIjogIkZ1bijwnZS5"
    "LCDihJUpIGlzIGNvdW50YWJsZTogYSBmdW5jdGlvbiDwnZS5IOKGkiDihJUgaXMgZGV0ZXJtaW5lZCBieSBmKHR0KSBhbmQgZihm"
    "ZikuIFNvIEZ1bijwnZS5LCDihJUpIOKJhSDihJUgw5cg4oSVIOKJhSDihJUgdmlhIM+GLiBCb3RoIHBhaXIgY29tcG9uZW50cyBp"
    "biDihJUg4oaSIGNvdW50YWJsZS4ifSwgIjE0IjogeyJhIjogIkIiLCAiZSI6ICJGdW4o4oSVLCDwnZS5KSBpcyBpbiBiaWplY3Rp"
    "b24gd2l0aCB0aGUgcG93ZXJzZXQgb2Yg4oSVIChlYWNoIGZ1bmN0aW9uID0gaW5kaWNhdG9yIGZ1bmN0aW9uIG9mIGEgc3Vic2V0"
    "KS4gfFAo4oSVKXwgPSAyXuKEteKCgCwgdW5jb3VudGFibGUuIn0sICIxNSI6IHsiYSI6ICJDIiwgImUiOiAiR2l2ZW4gKG0sIG4p"
    "IGFuZCAobScsIG4nKSB3aXRoIM+GKG0sIG4pID0gz4YobScsIG4nKSwgVW5pcXVlIFByaW1lIEZhY3Rvcml6YXRpb24gZ2l2ZXMg"
    "Ml5tID0gMl57bSd9IChzbyBtID0gbScpIGFuZCAybisxID0gMm4nKzEgKHNvIG4gPSBuJykuIFRoZSBwcm9vZiByZWxpZXMgb24g"
    "VVBGIHNwZWNpZmljYWxseS4ifSwgIjE2IjogeyJhIjogIkMiLCAiZSI6ICJTdGF0ZXMgaGF2ZSBvbmx5IGZpbml0ZWx5IG1hbnkg"
    "bm9uLXplcm8gdmFyaWFibGVzLiBXZSBjYW4gcmVwcmVzZW50IM+DIGFzIHRoZSBmaW5pdGUgbGlzdCBbz4MoeF8wKSwgz4MoeF8x"
    "KSwgLi4uXSB0cnVuY2F0ZWQgYXQgdGhlIGxhc3Qgbm9uLXplcm8uIFRoZW4gz4ggZW5jb2RlcyB0aGUgbGlzdC4ifSwgIjE3Ijog"
    "eyJhIjogIkIiLCAiZSI6ICJUaGUgdW5pdmVyc2FsIHByb2dyYW0gVSB0YWtlcyB0aGUgZW5jb2RlZCAocHJvZ3JhbSwgc3RhdGUp"
    "IHBhaXIsIGRlY29kZXMgdGhlIHByb2dyYW0gZnJvbSBpdHMgaW5kZXgsIGFuZCBzaW11bGF0ZXMgc3RlcC1ieS1zdGVwLiBMb3Rz"
    "IG9mIGNvZGUsIGJ1dCBtZWNoYW5pY2FsIGFuZCBmaW5pdGUg4oCUIGV4aXN0cyBpbiBwcmluY2lwbGUuIn0sICIxOCI6IHsiYSI6"
    "ICJCIiwgImUiOiAiRW5jb2RpbmcgaW50ZWdlciBhZGRpdGlvbjogZiA6IOKElSDDlyDihJUg4oaSIOKElSB0YWtlcyBlbmNvZGVk"
    "IGlucHV0cyAozrIoeCksIM6yKHkpKSBhbmQgcmV0dXJucyDOsih4K3kpLiBUaGUgZW5jb2RlZCB2ZXJzaW9uIGlzIGNvbXB1dGFi"
    "bGUgaW4gV2hpbGUuIFRoZSBkZWNvZGluZy1lbmNvZGluZyBwYXR0ZXJuIGxldHMgdXMgc3R1ZHkgYW55IGZ1bmN0aW9uIFMg4oaS"
    "IFQgdmlhIOKElSDihpIg4oSVLiJ9LCAiMTkiOiB7ImEiOiB0cnVlLCAiZSI6ICJTZXZlcmFsIGxpc3RzIGVuY29kZSB0aGUgc2Ft"
    "ZSBzdGF0ZSDigJQgZS5nLiwgWzAsIDBdIGFuZCBbXSBib3RoIHJlcHJlc2VudCB0aGUgYWxsLXplcm8gc3RhdGUuIFRoZSBlbmNv"
    "ZGluZyBoYXMgcmVkdW5kYW5jeTsgd2UganVzdCBwaWNrIGEgY2Fub25pY2FsIHJlcHJlc2VudGF0aXZlICh0eXBpY2FsbHk6IHN0"
    "cmlwIHRyYWlsaW5nIHplcm9zKS4ifSwgIjIwIjogeyJhIjogIkIiLCAiZSI6ICJCeSBEZWZpbml0aW9uIDE2LCBjb21wdXRhYmls"
    "aXR5IGZvciBmIDogUyDih4AgVCByZWR1Y2VzIHRvIHRoZSBlbmNvZGVkIGNvbXBvc2l0ZSDOtCDiiJggZiDiiJggzrMgYmVpbmcg"
    "Y29tcHV0YWJsZSBhcyDihJUg4oeAIOKElS4gQXMgbG9uZyBhcyB3ZSBoYXZlIGNvbXB1dGFibGUgaW5qZWN0aW9ucyDOsywgzrQs"
    "IHRoZSBjaG9pY2Ugb2Ygd2hpY2ggc2V0cyB0byBzdHVkeSBpcyBhIG5vbi1pc3N1ZS4ifX0="
)
_KEY = json.loads(base64.b64decode(_KEY_B64).decode('utf-8'))


def _print_explain(explain):
    if explain:
        for line in explain.split('\n'):
            print(f'   {line}')


def check(qnum, predicted):
    if predicted is Ellipsis:
        _RESULTS[qnum] = False
        print(f'Q{qnum}: ❌ Unanswered.')
        return
    info = _KEY[str(qnum)]
    actual = info['a']
    correct = predicted == actual
    _RESULTS[qnum] = correct
    if correct:
        print(f'Q{qnum}: ✅ Correct.')
    else:
        print(f'Q{qnum}: ❌ Incorrect.')
        print(f'  You answered: {predicted!r}')
        print(f'  Correct:      {actual!r}')
    _print_explain(info['e'])


print('quiz harness ready')


quiz harness ready


### Q1 — Cardinality of Fun(ℕ, ℕ)

Theorem 25 says Fun(ℕ, ℕ) is uncountable. The proof works by:

- **A.** Pigeonhole principle
- **B.** Cantor's diagonal: list any candidate enumeration, define g(n) = f_n(n) + 1, contradiction
- **C.** Compactness theorem
- **D.** Direct calculation


In [2]:
q1 = ...
check(1, q1)

Q1: ❌ Unanswered.


### Q2 — Why some functions are uncomputable

The combination of (a) at most countably many While programs and (b) Theorem 25 implies:

- **A.** Every function is computable
- **B.** No functions are computable
- **C.** Most functions ℕ → ℕ are uncomputable (uncountable - countable = uncountable)
- **D.** All computable functions terminate


In [3]:
q2 = ...
check(2, q2)

Q2: ❌ Unanswered.


### Q3 — Definition of β

The bijection β : ℤ → ℕ from §6.2.1 is defined as:

- **A.** β(x) = x if x ≥ 0, else 0
- **B.** β(x) = 2x if x ≥ 0, else -2x - 1
- **C.** β(x) = |x|
- **D.** β(x) = x²


In [4]:
q3 = ...
check(3, q3)

Q3: ❌ Unanswered.


### Q4 — Compute β(-3)

What is β(-3)?


In [5]:
q4 = ...   # an integer
check(4, q4)

Q4: ❌ Unanswered.


### Q5 — Definition of φ

The pairing function φ : ℕ × ℕ → ℕ from §6.2.2 is defined as:

- **A.** φ(m, n) = m + n
- **B.** φ(m, n) = m · n
- **C.** φ(m, n) = 2^m (2n + 1) - 1
- **D.** φ(m, n) = (m + n)(m + n + 1)/2 + n  (Cantor pairing)


In [6]:
q5 = ...
check(5, q5)

Q5: ❌ Unanswered.


### Q6 — Decode 40815 as a pair

What is the second component of φ⁻¹(40815)?  (The pair is (m, n) — give n.)


In [7]:
q6 = ...   # an integer
check(6, q6)

Q6: ❌ Unanswered.


### Q7 — Encoding tuples

How do we encode k-tuples (x₁, x₂, ..., x_k) as a single natural number?

- **A.** Concatenate digits
- **B.** Sum modulo p
- **C.** Right-fold of φ: (x₁, x₂, ..., x_k) ↦ φ(x₁, φ(x₂, φ(x₃, ..., x_k))...)
- **D.** Linear combination


In [8]:
q7 = ...
check(7, q7)

Q7: ❌ Unanswered.


### Q8 — Definition of ψ for lists

The list-encoding bijection ψ is defined as:

- **A.** ψ([x₁, ..., x_k]) = 2^x₁ · 3^x₂ · 5^x₃ · ... (Gödel-style)
- **B.** ψ([]) = 0; ψ(n :: l) = φ(n, ψ(l)) + 1
- **C.** ψ(l) = sum of elements
- **D.** ψ(l) = product of (1 + element)


In [9]:
q8 = ...
check(8, q8)

Q8: ❌ Unanswered.


### Q9 — Encoding skip

The Gödel number φ_S(skip) is:

- **A.** 5
- **B.** 1
- **C.** 0
- **D.** 4


In [10]:
q9 = ...
check(9, q9)

Q9: ❌ Unanswered.


### Q10 — Encoding assignment

For an assignment statement `x_i := a`, what does φ_S(x_i := a) look like?

- **A.** 1 + 4 · φ(φ_V(x_i), φ_A(a))
- **B.** φ_V(x_i) · φ_A(a)
- **C.** 4 · i + φ_A(a)
- **D.** φ(i, a)


In [11]:
q10 = ...
check(10, q10)

Q10: ❌ Unanswered.


### Q11 — Encoding numerals

For a numeral n in an arithmetic expression, φ_A(n) = ?

- **A.** n
- **B.** 5 · β(n)  (where β handles the integer-to-natural mapping since numerals can be negative)
- **C.** 1 + 5 · n
- **D.** β(n)


In [12]:
q11 = ...
check(11, q11)

Q11: ❌ Unanswered.


### Q12 — Practicality

Are Gödel numbers of programs practically usable as a serialization format?

- **A.** Yes — they're compact and easy to compute
- **B.** Yes — but slower than JSON
- **C.** No — even small programs produce astronomically large numbers; the encoding is a theoretical existence proof, not a practical format
- **D.** Yes for AExp/BExp, no for Stmt


In [13]:
q12 = ...
check(12, q12)

Q12: ❌ Unanswered.


### Q13 — Cardinality of Fun(𝔹, ℕ)

The set Fun(𝔹, ℕ) of functions from booleans to naturals is:

- **A.** Uncountable (by Cantor diagonal)
- **B.** Countable (a function 𝔹 → ℕ is determined by its two values, so Fun(𝔹, ℕ) ≅ ℕ × ℕ ≅ ℕ)
- **C.** Finite
- **D.** Empty


In [14]:
q13 = ...
check(13, q13)

Q13: ❌ Unanswered.


### Q14 — Cardinality of Fun(ℕ, 𝔹)

The set Fun(ℕ, 𝔹) is:

- **A.** Countable
- **B.** Uncountable — same diagonal argument; equivalent to the powerset of ℕ
- **C.** Finite
- **D.** Empty


In [15]:
q14 = ...
check(14, q14)

Q14: ❌ Unanswered.


### Q15 — Why φ is injective

The proof that φ : ℕ × ℕ → ℕ is injective uses:

- **A.** Cantor diagonal
- **B.** Pigeonhole
- **C.** Unique Prime Factorization (UPF) — 2^m and 2n+1 uniquely determine each other in any factorization
- **D.** Linear algebra


In [16]:
q15 = ...
check(15, q15)

Q15: ❌ Unanswered.


### Q16 — Encoding states

To encode a state σ as a natural number, we:

- **A.** Take σ as a Python dict and serialize to JSON
- **B.** Hash the variable names
- **C.** Build the list [σ(x_0), σ(x_1), ...] truncated to the last non-zero entry, then apply ψ
- **D.** Sum all variable values


In [17]:
q16 = ...
check(16, q16)

Q16: ❌ Unanswered.


### Q17 — The universal program U

The universal program U:

- **A.** Decides the halting problem
- **B.** Takes encoded (program, state) inputs and simulates the small-step semantics; chapter argues it exists informally
- **C.** Doesn't exist
- **D.** Only works on programs without loops


In [18]:
q17 = ...
check(17, q17)

Q17: ❌ Unanswered.


### Q18 — Encoding operations

To compute integer addition x + y where the inputs are β-encoded as naturals, we:

- **A.** Use a different encoding
- **B.** Define f : ℕ × ℕ → ℕ that takes β(x), β(y) and returns β(x + y) — the encoded version is computable in While
- **C.** Need to extend While with negative numbers
- **D.** Can't do this — encoding is one-way


In [19]:
q18 = ...
check(18, q18)

Q18: ❌ Unanswered.


### Q19 — State-encoding redundancy

True or False: the same state σ can be encoded by multiple different lists.


In [20]:
q19 = ...   # True or False
check(19, q19)

Q19: ❌ Unanswered.


### Q20 — Why Definition 16 matters

What's the practical role of Definition 16 (computability for f : S ⇀ T via injections)?

- **A.** It's just notation — makes no difference
- **B.** It lets us reduce computability questions for any reasonable type-pair (S, T) to the canonical case ℕ ⇀ ℕ via encodings — so chapter 5's results about ℕ ⇀ ℕ apply broadly
- **C.** It's only relevant for ℝ-valued functions
- **D.** It contradicts Definition 14


In [21]:
q20 = ...
check(20, q20)

Q20: ❌ Unanswered.


## Final score

Run this once you've answered all 20 questions.

In [22]:
total = len(_RESULTS)
correct = sum(_RESULTS.values())
missing = [n for n in range(1, 21) if n not in _RESULTS]

print(f'Score: {correct}/{total}')
if missing:
    print(f'  (you have not answered: {missing})')

if total == 0:
    print('No answers recorded.')
elif correct == 20:
    print('Perfect — encodings are yours.')
elif correct >= 18:
    print('Strong grasp.')
elif correct >= 15:
    print('Solid. Re-read the section(s) the missed questions came from.')
elif correct >= 12:
    print('Mid-pack. Worth one more pass through N15.')
else:
    print('Worth a full re-read. The β / φ / φ_S definitions are the core.')

print()
print('Question-by-question:')
section_map = {
    'Cardinality':              [1, 2, 13, 14],
    'Integer encoding (β)':     [3, 4],
    'Pair encoding (φ)':        [5, 6, 7, 15],
    'Lists and tuples (ψ)':     [8],
    'Program encoding (φ_A, φ_B, φ_S)': [9, 10, 11, 12],
    'States and operations':    [16, 17, 18, 19, 20],
}
for section, qnums in section_map.items():
    sec_correct = sum(_RESULTS.get(q, False) for q in qnums)
    sec_total = len(qnums)
    marks = ' '.join('OK' if _RESULTS.get(q) else ('X ' if q in _RESULTS else '. ') for q in qnums)
    print(f'  {section}: {sec_correct}/{sec_total}   {marks}')

Score: 0/20
Worth a full re-read. The β / φ / φ_S definitions are the core.

Question-by-question:
  Cardinality: 0/4   X  X  X  X 
  Integer encoding (β): 0/2   X  X 
  Pair encoding (φ): 0/4   X  X  X  X 
  Lists and tuples (ψ): 0/1   X 
  Program encoding (φ_A, φ_B, φ_S): 0/4   X  X  X  X 
  States and operations: 0/5   X  X  X  X  X 


## Where to go from here

If you missed questions in:
- **Cardinality** -> re-read N15 section 1 and Ex 53.
- **Encodings** -> re-read N15 sections 2-4 and the relevant exercises (54-56).
- **Program encoding** -> re-read N15 section 5 + Ex 57.
- **Universality** -> re-read N15 section 6 (and N13 if you want the bigger picture).